### 1. Import Dependencies

In [1]:
!pip install tensorflow


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install opencv-python


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip uninstall -y mediapipe numpy

!pip cache purge

!pip install numpy mediapipe --no-cache-dir --force-reinstall

print("Installed latest compatible versions. Please restart the runtime (Runtime -> Restart Runtime) before running code")

Found existing installation: mediapipe 0.10.32
Uninstalling mediapipe-0.10.32:
  Successfully uninstalled mediapipe-0.10.32
Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
Files removed: 260 (8.5 MB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----------------- ---------------------- 5.2/12.3 MB 31.7 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 40.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---------------------------------------- 10.2/10.2 MB 70.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 63.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 138.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import numpy as np
import os
from matplotlib import pyplot as plt
import time

import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks import python


### 2. Keypoints using MP Holistic

In [12]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Color conversion
    image.flags.writeable = False  # Image is no longer writable
    results = model.process(image)  # Make prediction
    image.flags.writeable = True  # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  # Color conversion
    return image, results

In [42]:
def draw_landmarks(frame, pose_result=None, hand_result=None, face_result=None):
    h, w, _ = frame.shape

    # ===== POSE =====
    if pose_result and pose_result.pose_landmarks:
        for person in pose_result.pose_landmarks:
            for lm in person:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)

    # ===== HANDS =====
    if hand_result and hand_result.hand_landmarks:
        for hand in hand_result.hand_landmarks:
            for lm in hand:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 4, (255, 0, 0), -1)

    # ===== FACE =====
    if face_result and face_result.face_landmarks:
        for face in face_result.face_landmarks:
            for lm in face:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 1, (0, 0, 255), -1)

    return frame

In [43]:
cap = cv2.VideoCapture(0)

pose_landmark = "pose_landmarker_lite.task"
hand_landmark = "hand_landmarker.task"
face_landmark = "face_landmarker.task"

# ===== POSE =====
pose_base = python.BaseOptions(model_asset_path=pose_landmark)
pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base,
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

# ===== HANDS =====
hand_base = python.BaseOptions(model_asset_path=hand_landmark)
hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

# ===== FACE =====
face_base = python.BaseOptions(model_asset_path=face_landmark)
face_options = vision.FaceLandmarkerOptions(
    base_options=face_base,
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

timestamp = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    # ===== DETECTIONS =====
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    hand_result = hand_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)

    timestamp += 1

    # # ===== PRINT DATA =====
    # print("POSE:", pose_result.pose_landmarks)
    # print("HANDS:", hand_result.hand_landmarks)
    # print("FACE:", face_result.face_landmarks)

    frame = draw_landmarks(frame, pose_result, hand_result, face_result)
    cv2.imshow('Feed', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [40]:
print(len(hand_result.))

IndexError: list index out of range

### 3. Extract Keypoint Values